# P8 — 네 층 READ TC 평가·개입 (학습 반환 감사 통과)

GPU 런타임에서 위부터 실행합니다. 새 결과는 Drive `boolean_interp_v1_4/P8_evaluation_r1`에 저장합니다. P5 원본 cache와 선택 TC 24개를 유지합니다. 먼저 모든 train/validation feature를 고정하고, test 의미 평가 및 한 위치 대체·인과 평가를 실행합니다. P8 완료는 반환 감사 후 판정합니다.

중단 후 같은 노트북을 실행하면 저장된 fit/선택/patch 단위부터 재개합니다. Colab 실제 GPU smoke는 새 환경마다 확인합니다. 모든 k·층·seed를 끝내며 결과가 나빠도 설정을 바꾸지 않습니다. 단계별 시간이 별도 기록됩니다. Test bins/candidate/feature를 재선택하지 않습니다.

입력 ZIP을 Drive `boolean_interp_v1_4/`에 업로드해 두세요. 24 TC는 이미 학습·반환 감사를 마쳤으며 이 노트북은 재학습하지 않습니다. 입력 u로 latent를 계산하고 출력 m을 s_m으로 대체·패칭하며 원본 residual skip을 유지합니다. Full u/full m과 m 좌표/random을 비교합니다. P7 h-SAE와 P8 m-TC의 서로 다른 target을 비교 그림에 명시합니다.

In [ ]:
from google.colab import files, drive
from pathlib import Path
import hashlib, zipfile, subprocess, sys, time, json, os
drive.mount('/content/drive')
BUNDLE_IN_DRIVE=Path('/content/drive/MyDrive/boolean_interp_v1_4/v1_4_p8_evaluation_r1.zip')
bundle=Path('/content/v1_4_p8_evaluation_r1.zip')
import shutil
def checksum(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(8*1024*1024),b''):h.update(b)
    return h.hexdigest()
if not bundle.exists() or checksum(bundle)!='3a122c4fb1e28113e0349c80aa26da88968c5f47997b4c9f5ab97fa9f82c1b66':shutil.copyfile(BUNDLE_IN_DRIVE,bundle)
assert checksum(bundle)=='3a122c4fb1e28113e0349c80aa26da88968c5f47997b4c9f5ab97fa9f82c1b66', '입력 ZIP checksum 불일치'
ROOT=Path('/content/MI_P8_evaluation_r1');ROOT.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:
    for i in z.infolist():assert (ROOT/i.filename).resolve().is_relative_to(ROOT.resolve())
    z.extractall(ROOT)
SOURCE=Path('/content/drive/MyDrive/boolean_interp_v1_4/P5_r2')
OUTPUT=Path('/content/drive/MyDrive/boolean_interp_v1_4/P8_evaluation_r1');OUTPUT.mkdir(parents=True,exist_ok=True)
assert (SOURCE/'contract.json').exists(), 'P5 원본 경로 확인 필요'


## 1. 의존성 및 checksum 검증
설치 후 재시작이 필요하면 재시작하고 업로드 셀부터 실행합니다. 설치 로그, pip freeze, GPU·환경 ID를 보존합니다.

In [ ]:
log=subprocess.run([sys.executable,'-m','pip','install','numpy==2.1.3','scipy==1.16.3','pytest==8.4.2','matplotlib==3.10.8','torch==2.11.0','--extra-index-url','https://download.pytorch.org/whl/cu128'],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(log.stdout);log.check_returncode();(OUTPUT/f'install_{time.time_ns()}.txt').write_text(log.stdout)
os.environ['OMP_NUM_THREADS']='2';os.environ['OPENBLAS_NUM_THREADS']='2'
sys.path.insert(0,str(ROOT))
from interp_v1_4.p8_evaluation import verify
verify(ROOT)
def stage(action, *extra):
    subprocess.run([sys.executable,'-m','interp_v1_4.p8_eval_runner',action,'--root',str(ROOT),'--source',str(SOURCE),'--output',str(OUTPUT),*extra],cwd=ROOT,check=True)
subprocess.run([sys.executable,'-m','pytest','tests_v1_4/test_p8_evaluation.py','-q'],cwd=ROOT,check=True)


## 2. 현재 GPU 코드 smoke
네 층에서 identity, mean, approximation, full donor, 선택 feature, random latent, 좌표, 두 random 방향 대조와 양방향·changed/unchanged 경로를 debug 모델로 검사합니다. Debug feature·결과는 본실험에 재사용하지 않습니다.

In [ ]:
SMOKE=OUTPUT/'smoke'/str(time.time_ns())
subprocess.run([sys.executable,'-m','interp_v1_4.p8_eval_runner','smoke','--root',str(ROOT),'--output',str(SMOKE),'--device','cuda'],cwd=ROOT,check=True)


## 3. 모든 train/validation 선택 동결
원본 240 cache checksum·READ key·라벨을 검사합니다. P5 full u/m probe와 block 0 m 좌표/random 대조군은 독립 감사된 계수를 재사용합니다. 다른 층의 좌표/random 및 TC 후보 전체·128개에서 라벨별 단일/≤4 probe를 선택합니다. CPU fitting은 시간이 걸릴 수 있습니다. 개별 fit마다 저장하며 중단 후 재개됩니다.

In [ ]:
stage('select','--device','cuda')


## 4. 동결된 의미 평가 및 fidelity
Test의 balanced accuracy/F1/AUROC/CI, 현재≠과거와 감독 변수 전이, NMSE/R2/EV·L0·train 전체 재인코딩 dead 비율을 저장합니다. READ의 연산 전이는 적용 대상이 아닙니다. Test로 feature를 바꾸지 않습니다.

In [ ]:
stage('semantic','--device','cuda')


## 5. 한 위치 대체·인과 평가
24개 순서대로 실행합니다. Causal validation의 norm bins를 먼저 저장합니다. Shared general test READ 2,048개를 층당 한 곳씩 대체하고, changed/unchanged 각 1,024 pair를 memory/composition과 양방향으로 평가합니다. Random latent 후보 최대 200개를 norm으로 분류하고 matched 최대 20개, unmatched는 사전 RNG 첫 후보를 평가합니다. Matching 실패를 NA·coverage로 보존합니다. 각 pair를 저장하므로 중단해도 완료 pair를 다시 평가하지 않습니다.

In [ ]:
stage('patch','--device','cuda','--smoke',str(SMOKE/'smoke.json'))


## 6. 표·그림·층간 paired CI 및 원시 결과 검산
모든 run이 있어야 집계합니다. 재현용 raw logits·norm·candidate·probe prediction을 함께 반환합니다. 검산은 저장된 logits에서 metric을 다시 계산하고 bin/feature/hash를 대조합니다. 전체 fitting·fidelity·CI의 독립 재현은 반환 후 cache 원본을 사용한 별도 감사 대상입니다.

In [ ]:
stage('aggregate')
subprocess.run([sys.executable,str(ROOT/'scripts/verify_v1_4_p8_evaluation.py'),'--root',str(ROOT),'--output',str(OUTPUT)],cwd=ROOT,check=True)


## 7. 결과 다운로드
완료 또는 중단 시 실행합니다. 아래 ZIP과 checksum JSON을 이 작업에 전달하면 실제 반환 감사를 진행합니다. P5 대형 cache와 입력 checkpoint는 ZIP에 다시 넣지 않습니다. 미완료 run도 실패/환경/재개 기록이 보존됩니다.

In [ ]:
archive=Path(f'/content/v1_4_p8_evaluation_return_{time.time_ns()}.zip')
stage('export','--archive',str(archive))
files.download(str(archive))
files.download(str(archive.with_suffix('.sha256.json')))
